# Panoramic X-ray to 3D scan: training (Kaggle)

This notebook trains a model that gets a panoramic X-ray as input and outputs a straightened 3D block of the jaw.
It reads the prepared scans from the private Kaggle dataset `xray-prepared-scans`.

**To start the full training:** the first code cell is already set for it (`EPOCHS = 150`, `TRAIN_SCANS = 0`, `RUN_NAME = "full"`).
Click "Save Version", choose "Save & Run All (Commit)", and click "Save".
Kaggle then runs the whole notebook on its own computers. You can close the browser and the laptop.

**To see the results:** open this notebook's page on Kaggle and open the "Output" tab of the newest version.
The files are in `runs/RUN_NAME/`: `best.pt` and `last.pt` (the trained model), `log.csv` (numbers for every epoch), and pictures.

**If a run stops before the end:** run again with the same `RUN_NAME`. The training continues after the last saved epoch.
This works because this notebook's own output is added as an input (see the "Input" panel).

In [ ]:
EPOCHS = 150          # how many passes over the training scans. 150 is the full run. Use 1 for a short test.
TRAIN_SCANS = 0       # 0 = use all training scans (the full run). Use 2 for a short test.
RUN_NAME = "full"     # name of the results folder. Use "test" for a short test.

In [ ]:
import torch
assert torch.cuda.is_available(), "No graphics card. In the right panel choose Session options > Accelerator > GPU T4 x2, then run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import glob, os, shutil
files = sorted(glob.glob("/kaggle/input/**/*.npz", recursive=True))
assert files, "No prepared scans were found. Click 'Add Input' and add the dataset xray-prepared-scans."
os.makedirs("/tmp/prepared", exist_ok=True)
for f in files:
    shutil.copy(f, "/tmp/prepared/")
print(len(files), "prepared scans copied")

OUT = "/kaggle/working/runs/" + RUN_NAME
# Saved work of an earlier run. It is only found if that run's output was added to this notebook as an input.
earlier = sorted(glob.glob("/kaggle/input/**/runs/" + RUN_NAME + "/last.pt", recursive=True))
if earlier and not os.path.exists(OUT + "/last.pt"):
    shutil.copytree(os.path.dirname(earlier[-1]), OUT, dirs_exist_ok=True)
    print("copied the saved work of an earlier run:", earlier[-1])
else:
    print("no saved work of an earlier run was found")

In [ ]:
os.makedirs("/kaggle/working/project/src", exist_ok=True)
os.makedirs("/kaggle/working/project/scripts", exist_ok=True)

The next 7 cells write the project's code files into Kaggle. They are copies of the files in the project folder.

In [ ]:
%%writefile /kaggle/working/project/src/fake_panoramic.py
"""Make a fake panoramic X-ray from a 3D CBCT.

How it works, in plain steps:
1. Find the curve of the jaw, seen from above (the U shape), at the height of the tooth roots.
2. Walk along that curve. At each point, look straight across the curve, through the jaw.
3. Add up how much X-ray each bit of tissue along that short line would block. Bone blocks a lot, air almost none.
   Tissue near the curve counts fully; tissue further away counts less (real machines blur it out).
4. Each point on the curve becomes one column of the picture; each height in the scan becomes one row.

Volumes are indexed [z, y, x]. z grows toward the top of the head, y grows toward the back of the head.
"""
import numpy as np
from scipy.ndimage import binary_opening, gaussian_filter1d, label, map_coordinates, zoom

MU_WATER = 0.02  # how much 1 mm of water blocks X-rays (per mm), a typical value near 70 keV


def hu_to_mu(vol_hu: np.ndarray, metal_cap: float = 3000.0) -> np.ndarray:
    """Turn CT numbers (Hounsfield units) into how much each mm of tissue blocks X-rays."""
    hu = np.clip(vol_hu, -1000, metal_cap)
    return MU_WATER * (1 + hu / 1000.0)


def resample(vol: np.ndarray, spacing, new_spacing: float) -> np.ndarray:
    return zoom(vol, [s / new_spacing for s in spacing], order=1)


def find_bite_level(vol_hu: np.ndarray, teeth_hu: float = 1800) -> int:
    """Slice index where the most very bright voxels (tooth enamel, fillings) are: roughly where the teeth meet."""
    counts = (vol_hu > teeth_hu).sum(axis=(1, 2))
    return int(np.argmax(gaussian_filter1d(counts.astype(float), 3)))


def find_arch_curve(vol_hu: np.ndarray, z_lo: int, z_hi: int, bone_hu: float = 500, metal_hu: float = 2800,
                    angle_range_deg: float = 100, n_angles: int = 201):
    """Find the jaw's U shape, seen from above, between slices z_lo and z_hi.

    Keeps bone but not metal, keeps the biggest connected piece (the jaw), then stands at a point
    inside the U and looks outward in many directions. Along each direction, the middle of the
    last stretch of bone before the outside of the face is one point of the curve.
    The distances are smoothed so the curve has no jumps.
    Returns (points_x, points_y, center).
    """
    slab = vol_hu[z_lo:z_hi]
    bone = ((slab > bone_hu) & (slab < metal_hu)).any(axis=0)
    bone = binary_opening(bone, iterations=2)
    lab, _ = label(bone)
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    bone = lab == sizes.argmax()
    return arch_curve_from_shape(bone, angle_range_deg, n_angles)


def arch_curve_from_shape(bone: np.ndarray, angle_range_deg: float = 100, n_angles: int = 201):
    """The second half of find_arch_curve. `bone` is a yes/no picture of the jaw seen from above."""
    ys, xs = np.nonzero(bone)
    cx = (xs.min() + xs.max()) / 2
    cy = ys.max() - 0.15 * (ys.max() - ys.min())  # inside the U, near the back ends of the jaw
    angles = np.deg2rad(np.linspace(-angle_range_deg, angle_range_deg, n_angles))  # 0 = straight to the front
    r_max = np.hypot(*bone.shape)
    radii = np.arange(0, r_max, 0.5)
    r_mid = np.full(len(angles), np.nan)
    for i, a in enumerate(angles):
        dx, dy = np.sin(a), -np.cos(a)
        px, py = cx + radii * dx, cy + radii * dy
        inside = (px >= 0) & (px < bone.shape[1] - 1) & (py >= 0) & (py < bone.shape[0] - 1)
        hit = np.zeros(len(radii), bool)
        hit[inside] = bone[np.round(py[inside]).astype(int), np.round(px[inside]).astype(int)]
        idx = np.flatnonzero(hit)
        if idx.size == 0:
            continue
        end = idx[-1]
        start = end
        while start - 1 >= 0 and hit[start - 1]:
            start -= 1
        r_mid[i] = radii[(start + end) // 2]
    ok = ~np.isnan(r_mid)
    # smooth the distance-vs-direction curve with a low-order polynomial, dropping outliers
    keep = ok.copy()
    for _ in range(3):
        coef = np.polyfit(angles[keep], r_mid[keep], 6)
        res = np.abs(r_mid - np.polyval(coef, angles))
        keep = ok & (res < max(3 * np.nanmedian(res[keep]), 2.0))
    a_used = angles[ok]
    a_fine = np.linspace(a_used.min(), a_used.max(), 2000)
    r_fine = np.polyval(coef, a_fine)
    return cx + r_fine * np.sin(a_fine), cy - r_fine * np.cos(a_fine), (cx, cy), (angles[keep], r_mid[keep])


def curve_points(cx_pts, cy_pts, center, step: float):
    """Points spaced `step` voxels apart along the curve, plus the direction across the curve (pointing outward)."""
    seg = np.hypot(np.diff(cx_pts), np.diff(cy_pts))
    arc = np.concatenate([[0], np.cumsum(seg)])
    s = np.arange(0, arc[-1], step)
    px, py = np.interp(s, arc, cx_pts), np.interp(s, arc, cy_pts)
    tx, ty = np.gradient(px), np.gradient(py)
    nx, ny = ty, -tx
    norm = np.hypot(nx, ny)
    nx, ny = nx / norm, ny / norm
    flip = (px - center[0]) * nx + (py - center[1]) * ny < 0
    nx[flip] *= -1; ny[flip] *= -1
    return px, py, nx, ny


def make_panoramic(mu: np.ndarray, curve_x, curve_y, center, voxel_mm: float,
                   depth_in_mm: float = 150, depth_out_mm: float = 30, sharp_mm: float = 4,
                   tilt_deg: float = 7, near_mm: float = 25, near_blur_px: float = 6,
                   far_blur_px: float = 25) -> np.ndarray:
    """Return the fake panoramic picture: rows = height (top of head first), columns = along the jaw
    (patient's right side on the left of the picture, like a real panoramic X-ray).

    Each column's light line starts outside the face and goes depth_in_mm into the head, so it
    passes through the whole head (tongue, throat, spine), as in a real machine.
    The line tilts upward by tilt_deg toward the front, like a real machine's beam.
    Everything along the line blocks light, but only tissue within about sharp_mm of the jaw curve is
    sharp. Tissue up to near_mm away is smeared sideways by near_blur_px columns, and tissue further
    away (for example the spine) by far_blur_px, because the machine moves while taking the picture.
    """
    px, py, nx, ny = curve_points(curve_x, curve_y, center, step=1.0)
    t = np.arange(-depth_in_mm, depth_out_mm, voxel_mm) / voxel_mm  # negative = into the head
    d = np.abs(t * voxel_mm)
    w_sharp = np.exp(-0.5 * (d / sharp_mm) ** 2)
    w_near = (1 - w_sharp) * (d <= near_mm)
    w_far = (1 - w_sharp) * (d > near_mm)
    dz = t * np.tan(np.deg2rad(tilt_deg))  # points deeper in the head sit lower on the line
    nz = mu.shape[0]
    sharp, near, far = (np.zeros((nz, len(px)), np.float32) for _ in range(3))
    zz = np.arange(nz)[:, None] + dz[None, :]
    for i in range(len(px)):
        sx = np.broadcast_to(px[i] + t * nx[i], zz.shape)
        sy = np.broadcast_to(py[i] + t * ny[i], zz.shape)
        vals = map_coordinates(mu, np.stack([zz.ravel(), sy.ravel(), sx.ravel()]), order=1, cval=0.0)
        vals = vals.reshape(zz.shape) * voxel_mm
        sharp[:, i] = vals @ w_sharp
        near[:, i] = vals @ w_near
        far[:, i] = vals @ w_far
    img = sharp + gaussian_filter1d(near, near_blur_px, axis=1) + gaussian_filter1d(far, far_blur_px, axis=1)
    return img[::-1]


def look_like_real(img: np.ndarray, clip_limit: float = 1.2, sharpen: float = 0.6) -> np.ndarray:
    """Imitate the picture clean-up that panoramic machines do: gentle local brightness balancing
    and edge sharpening. Returns values between 0 and 1."""
    import cv2
    lo, hi = np.percentile(img, [0.5, 99.8])
    x = np.clip((img - lo) / (hi - lo), 0, 1)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(4, 8))
    x = clahe.apply((x * 65535).astype(np.uint16)).astype(np.float32) / 65535
    blurred = cv2.GaussianBlur(x, (0, 0), 3)
    return np.clip(x + sharpen * (x - blurred), 0, 1)


In [ ]:
%%writefile /kaggle/working/project/src/tf3_io.py
"""Load ToothFairy3 scans and label files.

Every array returned here is indexed [z, y, x], the same as the rest of the code:
z grows toward the top of the head, y grows toward the back of the head, x grows toward the patient's left.
"""
from pathlib import Path

import nibabel as nib
import numpy as np

# The numbers the model predicts. Left and right parts are joined into one number.
CLASS_NAMES = ["background", "lower jaw", "upper jaw", "nerve canal", "sinus", "teeth"]
N_CLASSES = len(CLASS_NAMES)


def _to_zyx(img) -> np.ndarray:
    # as_closest_canonical gives [x toward right, y toward front, z toward top]; turn that into our order
    arr = np.asarray(nib.as_closest_canonical(img).dataobj)
    return np.ascontiguousarray(arr.transpose(2, 1, 0)[:, ::-1, ::-1])


def load_scan(path) -> np.ndarray:
    """The 3D scan in Hounsfield units, float32."""
    return _to_zyx(nib.load(str(path))).astype(np.float32)


def load_labels(path) -> np.ndarray:
    """The label file with the original ToothFairy3 numbers."""
    return _to_zyx(nib.load(str(path))).astype(np.int16)


def merge_labels(lab: np.ndarray) -> np.ndarray:
    """Turn the ToothFairy3 numbers into the 6 numbers in CLASS_NAMES.

    ToothFairy3 numbers: 1 lower jaw, 2 upper jaw, 3 and 4 nerve canal, 5 and 6 sinus, 7 throat,
    8 bridge, 9 crown, 10 implant, 11 to 48 single teeth, 103 to 105 small canals at the front of the
    lower jaw, 111 to 148 the soft inside of each tooth.
    """
    out = np.zeros(lab.shape, np.uint8)
    out[lab == 1] = 1
    out[(lab >= 103) & (lab <= 105)] = 1  # small front canals are inside the lower jaw bone
    out[lab == 2] = 2
    out[(lab == 3) | (lab == 4)] = 3
    out[(lab == 5) | (lab == 6)] = 4
    out[((lab >= 8) & (lab <= 48)) | ((lab >= 111) & (lab <= 148))] = 5
    return out


def list_scans(data_dir):
    """Pairs of (scan number as text, scan file, label file) for every scan in the folder."""
    data_dir = Path(data_dir)
    pairs = []
    for scan in sorted((data_dir / "imagesTr").glob("ToothFairy3F_*_0000.nii.gz")):
        num = scan.name.split("_")[1]
        pairs.append((num, scan, data_dir / "labelsTr" / f"ToothFairy3F_{num}.nii.gz"))
    return pairs


In [ ]:
%%writefile /kaggle/working/project/src/curved_volume.py
"""Turn a 3D scan into a "straightened" 3D block that lines up with the fake panoramic X-ray.

The fake X-ray has one column for each point along the U-shaped jaw curve and one row for each height.
The straightened block has the same rows and columns, plus a third direction: how far a point is from
the jaw curve, measured straight across the curve (toward the tongue or toward the cheek).

So pixel (row, column) of the X-ray and the line block[row, column, :] show the same place in the head.
The model gets the X-ray as input and must output the block.

Arrays are indexed [z, y, x] for scans and [row, column, depth] for blocks. Row 0 is the top of the head.
"""
import numpy as np
from scipy.ndimage import map_coordinates

from .fake_panoramic import arch_curve_from_shape, curve_points, find_arch_curve, find_bite_level

VOXEL_MM = 0.4      # size of one voxel in the scans, and of one row or column in the X-ray and block
DEPTH_IN_MM = 22.0  # how far the block reaches from the jaw curve toward the tongue
DEPTH_OUT_MM = 10.0 # how far the block reaches from the jaw curve toward the cheek
DEPTH_STEP_MM = 0.5 # distance between two neighbouring depth samples
OUTSIDE = 255       # label value for places that are outside the scan


def depth_positions_mm() -> np.ndarray:
    """Distance from the jaw curve of each depth sample. Negative = toward the tongue."""
    return np.arange(-DEPTH_IN_MM, DEPTH_OUT_MM, DEPTH_STEP_MM)


def jaw_curve(vol_hu: np.ndarray):
    """Find the jaw curve of a scan. Returns (bite slice, a dict with the curve)."""
    bite = find_bite_level(vol_hu)
    z_lo, z_hi = max(bite - int(20 / VOXEL_MM), 0), bite + int(5 / VOXEL_MM)
    cx, cy, center, _ = find_arch_curve(vol_hu, z_lo, z_hi)
    px, py, nx, ny = curve_points(cx, cy, center, step=1.0)
    middle = int(np.argmin(np.abs(px - center[0]) + 1e3 * (py > center[1])))  # the front-most point of the U
    return bite, dict(cx=cx, cy=cy, center=np.array(center), px=px, py=py, nx=nx, ny=ny, middle=middle)


def straighten(vol: np.ndarray, curve: dict, is_label: bool):
    """Sample the scan (or label file) along the jaw curve. Returns block[row, column, depth].

    For a scan, places outside the scan get -1000 (air). For labels they get OUTSIDE.
    """
    t = depth_positions_mm() / VOXEL_MM
    sx = curve["px"][:, None] + t[None, :] * curve["nx"][:, None]
    sy = curve["py"][:, None] + t[None, :] * curve["ny"][:, None]
    coords = np.stack([sy, sx])
    out = np.empty((vol.shape[0], sx.shape[0], sx.shape[1]), np.uint8 if is_label else np.float32)
    for z in range(vol.shape[0]):
        if is_label:
            out[z] = map_coordinates(vol[z], coords, order=0, cval=OUTSIDE)
        else:
            out[z] = map_coordinates(vol[z], coords, order=1, cval=-1000.0)
    return out[::-1]


def bend_back(block: np.ndarray, curve: dict, shape_zyx, fill: float, is_label: bool = False) -> np.ndarray:
    """The opposite of straighten: put a block back into the normal [z, y, x] shape of a scan.

    For each (y, x) place it finds the nearest point on the jaw curve and the distance across the curve,
    then reads the block there. Places further from the curve than the block reaches get `fill`.
    """
    from scipy.spatial import cKDTree
    nz, ny_, nx_ = shape_zyx
    yy, xx = np.mgrid[0:ny_, 0:nx_].astype(np.float32)
    pts = np.stack([curve["px"], curve["py"]], axis=1)
    _, idx = cKDTree(pts).query(np.stack([xx.ravel(), yy.ravel()], axis=1))
    dx, dy = xx.ravel() - curve["px"][idx], yy.ravel() - curve["py"][idx]
    # distance across the curve (signed), and distance along the curve from the nearest curve point
    across = dx * curve["nx"][idx] + dy * curve["ny"][idx]
    along = dx * np.gradient(curve["px"])[idx] + dy * np.gradient(curve["py"])[idx]
    col = idx + along  # curve points are 1 voxel apart, so 1 voxel along the curve = 1 column
    dep = (across * VOXEL_MM + DEPTH_IN_MM) / DEPTH_STEP_MM
    n_col, n_dep = block.shape[1], block.shape[2]
    ok = (col >= 0) & (col <= n_col - 1) & (dep >= 0) & (dep <= n_dep - 1) & (np.abs(along) < 3)
    out = np.full((nz, ny_ * nx_), fill, block.dtype)
    coords = np.stack([col[ok], dep[ok]])
    for z in range(nz):
        out[z, ok] = map_coordinates(block[nz - 1 - z], coords, order=0 if is_label else 1, mode="nearest")
    return out.reshape(nz, ny_, nx_)


# Every X-ray and block is put on a frame of one fixed size, so the model always gets the same size.
# The row where the teeth meet and the column of the front of the jaw are always at the same place.
FRAME_ROWS = 240
FRAME_COLS = 544
FRAME_BITE_ROW = 80
FRAME_MIDDLE_COL = FRAME_COLS // 2


def to_frame(arr: np.ndarray, bite_row: int, middle_col: int, fill):
    """Copy arr[row, column, ...] onto the fixed frame. Parts that do not fit are dropped.
    Returns (frame, row shift, column shift), where frame[r + row shift, c + column shift] = arr[r, c]."""
    dr, dc = FRAME_BITE_ROW - bite_row, FRAME_MIDDLE_COL - middle_col
    out = np.full((FRAME_ROWS, FRAME_COLS) + arr.shape[2:], fill, arr.dtype)
    r0, r1 = max(0, -dr), min(arr.shape[0], FRAME_ROWS - dr)
    c0, c1 = max(0, -dc), min(arr.shape[1], FRAME_COLS - dc)
    out[r0 + dr:r1 + dr, c0 + dc:c1 + dc] = arr[r0:r1, c0:c1]
    return out, dr, dc


def jaw_curve_from_labels(lab: np.ndarray):
    """Find the bite slice and the jaw curve of a training scan from its label file
    (the original ToothFairy3 numbers, not the merged ones).

    This is more reliable than jaw_curve, which has to guess from brightness alone.
    Bite slice = halfway between the tips of the lower teeth and the tips of the upper teeth.
    Jaw curve = the U shape of the lower jaw bone and the teeth, seen from above, from 30 mm below
    the bite slice to 5 mm above it.
    """
    lower = ((lab >= 31) & (lab <= 48)) | ((lab >= 131) & (lab <= 148))
    upper = ((lab >= 11) & (lab <= 28)) | ((lab >= 111) & (lab <= 128))
    tips = []
    for teeth, q in ((lower, 97), (upper, 3)):
        per_slice = teeth.sum(axis=(1, 2))
        if per_slice.sum() > 0:
            tips.append(np.searchsorted(np.cumsum(per_slice) / per_slice.sum(), q / 100))
    if not tips:
        raise ValueError("no teeth in the label file")
    bite = int(round(np.mean(tips)))
    z_lo, z_hi = max(bite - int(30 / VOXEL_MM), 0), min(bite + int(5 / VOXEL_MM), lab.shape[0])
    slab = lab[z_lo:z_hi]
    shape = ((slab == 1) | ((slab >= 8) & (slab <= 48)) | ((slab >= 111) & (slab <= 148))).any(axis=0)
    return bite, curve_from_shape(shape)


def curve_from_shape(shape: np.ndarray) -> dict:
    """The jaw curve from a yes/no picture of the lower jaw bone and the teeth, seen from above."""
    cx, cy, center, _ = arch_curve_from_shape(shape)
    # keep only the part of the curve that goes outward from the front; drop ends that turn back
    angle = np.arctan2(cx - center[0], center[1] - cy)
    radius = np.hypot(cx - center[0], cy - center[1])
    front = int(np.argmin(np.abs(angle) + 10 * (radius < 0.5 * np.median(radius))))
    good = radius > 0.35 * radius[front]
    # also drop ends that turn back toward the front, and ends that are past the last bit of jaw
    for side in (slice(front, None), slice(front, None, -1)):
        y = cy[side]
        good[side] &= np.cumsum(y < np.maximum.accumulate(y) - 1.0) == 0
    from scipy.ndimage import binary_dilation
    near_jaw = binary_dilation(shape, iterations=3)
    on_jaw = np.flatnonzero(near_jaw[np.clip(np.round(cy).astype(int), 0, shape.shape[0] - 1),
                                     np.clip(np.round(cx).astype(int), 0, shape.shape[1] - 1)])
    good[:on_jaw[0]] = False
    good[on_jaw[-1] + 1:] = False
    first = front - np.argmax(~good[front::-1]) + 1 if (~good[:front]).any() else 0
    last = front + np.argmax(~good[front:]) if (~good[front:]).any() else len(cx)
    cx, cy = cx[first:last], cy[first:last]
    px, py, nx, ny = curve_points(cx, cy, center, step=1.0)
    middle = int(np.argmin(np.hypot(px - cx[front - first], py - cy[front - first])))
    return dict(cx=cx, cy=cy, center=np.array(center), px=px, py=py, nx=nx, ny=ny, middle=middle)


def jaw_curve_from_brightness(vol_hu: np.ndarray, bite: int, hard_hu: float = 900):
    """The jaw curve of a scan that has no label file, found the same way as for the training scans:
    everything hard (bone, teeth, fillings) from 30 mm below the bite slice to 5 mm above it, seen
    from above, is used in place of the labels. The biggest connected piece is the jaw."""
    from scipy.ndimage import binary_fill_holes, binary_opening, label
    z_lo, z_hi = max(bite - int(30 / VOXEL_MM), 0), min(bite + int(5 / VOXEL_MM), vol_hu.shape[0])
    shape = binary_opening((vol_hu[z_lo:z_hi] > hard_hu).any(axis=0), iterations=2)
    pieces, _ = label(shape)
    sizes = np.bincount(pieces.ravel()); sizes[0] = 0
    return curve_from_shape(binary_fill_holes(pieces == sizes.argmax()))


In [ ]:
%%writefile /kaggle/working/project/src/model.py
"""The model: gets a panoramic X-ray, outputs the straightened 3D block.

Plain description:
1. A "U-Net" looks at the X-ray. A U-Net is a common program for pictures. It shrinks the picture
   step by step to find large shapes (a whole jaw), then grows it back step by step to find small
   shapes (one tooth root), and at each size it reuses what it found on the way down.
2. For each X-ray pixel the U-Net outputs a list of numbers. The list is turned into a short line
   of points going from the tongue side to the cheek side. All lines together are the 3D block.
3. A small 3D step looks at each point together with its neighbours in all three directions and
   outputs, for each point: how bright it is in a 3D scan, and which part it is (bone, tooth, ...).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def conv_block(n_in, n_out):
    return nn.Sequential(
        nn.Conv2d(n_in, n_out, 3, padding=1), nn.GroupNorm(8, n_out), nn.ReLU(inplace=True),
        nn.Conv2d(n_out, n_out, 3, padding=1), nn.GroupNorm(8, n_out), nn.ReLU(inplace=True))


class XrayTo3D(nn.Module):
    def __init__(self, in_channels=1, depth=64, n_classes=6, widths=(48, 96, 192, 384, 512), lift=8):
        """in_channels: 1 for the X-ray alone. More when extra inputs (from the mouth video) are added.
        depth: number of points on each tongue-to-cheek line."""
        super().__init__()
        self.depth, self.n_classes, self.lift = depth, n_classes, lift
        self.down = nn.ModuleList()
        n = in_channels
        for w in widths:
            self.down.append(conv_block(n, w))
            n = w
        self.up = nn.ModuleList()
        for w in reversed(widths[:-1]):
            self.up.append(conv_block(n + w, w))
            n = w
        self.to_lines = nn.Conv2d(n, lift * depth, 1)
        self.head3d = nn.Sequential(
            nn.Conv3d(lift, 24, 3, padding=1), nn.GroupNorm(8, 24), nn.ReLU(inplace=True),
            nn.Conv3d(24, 24, 3, padding=1), nn.GroupNorm(8, 24), nn.ReLU(inplace=True),
            nn.Conv3d(24, 1 + n_classes, 1))

    def forward(self, xray):
        """xray: [batch, in_channels, rows, columns], rows and columns must be multiples of 16.
        Returns (brightness [batch, depth, rows, columns] between 0 and 1,
                 part scores [batch, n_classes, depth, rows, columns])."""
        kept = []
        x = xray
        for i, block in enumerate(self.down):
            x = block(x)
            if i < len(self.down) - 1:
                kept.append(x)
                x = F.max_pool2d(x, 2)
        for block in self.up:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            x = block(torch.cat([x, kept.pop()], dim=1))
        b, _, h, w = x.shape
        x = self.to_lines(x).view(b, self.lift, self.depth, h, w)
        x = self.head3d(x)
        return torch.sigmoid(x[:, 0]), x[:, 1:]


In [ ]:
%%writefile /kaggle/working/project/src/train_data.py
"""Load the prepared examples and cut random training pieces from them."""
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

OUTSIDE = 255
HU_LOW, HU_HIGH = -1000.0, 3000.0


def hu_to_unit(hu):
    """Hounsfield units -> numbers between 0 and 1 (what the model outputs)."""
    return (hu - HU_LOW) / (HU_HIGH - HU_LOW)


def unit_to_hu(x):
    return x * (HU_HIGH - HU_LOW) + HU_LOW


def make_split(folder, n_check=5, n_test=10, seed=0):
    """Split the scans into train / check / test. Saved in split.json so it never changes.
    "check" scans are used during training to see if the model is getting better.
    "test" scans are never used during training."""
    folder = Path(folder)
    path = folder / "split.json"
    if path.exists():
        return json.loads(path.read_text())
    nums = sorted(p.stem for p in folder.glob("*.npz"))
    random.Random(seed).shuffle(nums)
    split = dict(test=sorted(nums[:n_test]), check=sorted(nums[n_test:n_test + n_check]),
                 train=sorted(nums[n_test + n_check:]))
    path.write_text(json.dumps(split, indent=1))
    return split


class Examples:
    """Holds a list of prepared scans in memory.
    xray: [n, rows, columns] 0..1,  hu: [n, depth, rows, columns] Hounsfield units,
    labels: [n, depth, rows, columns] part numbers, 255 = outside the scan."""

    def __init__(self, folder, numbers):
        xs, hs, ls = [], [], []
        for num in numbers:
            with np.load(Path(folder) / f"{num}.npz") as f:
                xs.append(torch.from_numpy(f["xray"].astype(np.float32)))
                hs.append(torch.from_numpy(f["hu"]).permute(2, 0, 1).contiguous())
                ls.append(torch.from_numpy(f["labels"]).permute(2, 0, 1).contiguous())
        self.numbers = list(numbers)
        self.xray, self.hu, self.labels = torch.stack(xs), torch.stack(hs), torch.stack(ls)

    def __len__(self):
        return len(self.numbers)

    def whole(self, i):
        """One whole example, as a batch of 1."""
        return self.xray[i:i + 1, None], self.hu[i:i + 1], self.labels[i:i + 1]

    def random_batch(self, batch_size, piece_cols, max_shift_rows=8):
        """Random pieces for training. Each piece is piece_cols columns wide and is randomly
        mirrored left-right and moved up or down a little, so the model sees more variety."""
        rows, cols = self.xray.shape[1:]
        xs, hs, ls = [], [], []
        for _ in range(batch_size):
            i = random.randrange(len(self))
            c = random.randrange(0, cols - piece_cols + 1)
            x, h, l = self.xray[i, :, c:c + piece_cols], self.hu[i, :, :, c:c + piece_cols], self.labels[i, :, :, c:c + piece_cols]
            if random.random() < 0.5:
                x, h, l = x.flip(-1), h.flip(-1), l.flip(-1)
            s = random.randint(-max_shift_rows, max_shift_rows)
            if s != 0:
                x, h, l = _shift_rows(x, s, 0.0), _shift_rows(h, s, int(HU_LOW)), _shift_rows(l, s, OUTSIDE)
            xs.append(x); hs.append(h); ls.append(l)
        return torch.stack(xs)[:, None], torch.stack(hs), torch.stack(ls)


def _shift_rows(t, s, fill):
    """Move the picture s rows down (or up if s is negative). Rows that become empty get `fill`."""
    out = torch.full_like(t, fill)
    if s > 0:
        out[..., s:, :] = t[..., :-s, :]
    else:
        out[..., :s, :] = t[..., -s:, :]
    return out


def change_xray_look(xray):
    """Randomly change brightness curve, contrast, blur and noise of a batch of X-rays [b, 1, rows, columns].
    Real X-rays do not look exactly like our fake ones, so the model must not depend on the exact look."""
    b = xray.shape[0]
    dev = xray.device
    gamma = torch.empty(b, 1, 1, 1, device=dev).uniform_(0.7, 1.5)
    x = xray.clamp(0, 1) ** gamma
    contrast = torch.empty(b, 1, 1, 1, device=dev).uniform_(0.75, 1.25)
    bright = torch.empty(b, 1, 1, 1, device=dev).uniform_(-0.08, 0.08)
    x = (x - 0.5) * contrast + 0.5 + bright
    if random.random() < 0.5:
        sigma = random.uniform(0.5, 2.0)
        k = torch.arange(-6, 7, device=dev, dtype=x.dtype)
        k = torch.exp(-0.5 * (k / sigma) ** 2)
        k = k / k.sum()
        x = F.conv2d(F.pad(x, (6, 6, 0, 0), mode="replicate"), k.view(1, 1, 1, -1))
        x = F.conv2d(F.pad(x, (0, 0, 6, 6), mode="replicate"), k.view(1, 1, -1, 1))
    x = x + torch.randn_like(x) * random.uniform(0.0, 0.04)
    return x.clamp(0, 1)


In [ ]:
%%writefile /kaggle/working/project/scripts/prepare_tf3.py
"""Turn every ToothFairy3 scan into one training example.

For each scan this makes:
  - the fake panoramic X-ray (what the model gets as input)
  - the straightened 3D block of the scan and of the labels (what the model must output)
and saves them in one file per scan, plus one small picture per scan to look at.

Run:  python scripts/prepare_tf3.py --data data/tf3F_0.4mm --out data/prepared
"""
import argparse
import json
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src import curved_volume as cv
from src.fake_panoramic import hu_to_mu, look_like_real, make_panoramic
from src.tf3_io import CLASS_NAMES, list_scans, load_labels, load_scan, merge_labels

MIN_SLICES = 150       # scans shorter than this (60 mm) do not hold the whole jaw and are skipped
MIN_ROWS_ABOVE = 30    # scans with less than this (12 mm) above the bite do not hold the upper teeth and are skipped
LABEL_COLORS = np.array([[0, 0, 0], [70, 110, 200], [60, 170, 170], [255, 40, 40], [255, 220, 0], [240, 240, 240]], np.uint8)


def prepare_one(scan_path, label_path):
    vol = load_scan(scan_path)
    lab_original = load_labels(label_path)
    lab = merge_labels(lab_original)
    if vol.shape != lab.shape:
        raise ValueError(f"scan {vol.shape} and labels {lab.shape} have different sizes")
    if vol.shape[0] < MIN_SLICES:
        raise ValueError(f"only {vol.shape[0]} slices")
    bite, curve = cv.jaw_curve_from_labels(lab_original)
    if vol.shape[0] - 1 - bite < MIN_ROWS_ABOVE:
        raise ValueError(f"only {vol.shape[0] - 1 - bite} slices above the bite")
    xray = look_like_real(make_panoramic(hu_to_mu(vol), curve["cx"], curve["cy"], curve["center"], cv.VOXEL_MM))
    hu = cv.straighten(vol, curve, is_label=False)
    labels = cv.straighten(lab, curve, is_label=True)
    bite_row = vol.shape[0] - 1 - bite  # row 0 is the top of the head
    xray_f, dr, dc = cv.to_frame(xray.astype(np.float32), bite_row, curve["middle"], 0.0)
    hu_f, _, _ = cv.to_frame(np.clip(hu, -1000, 3000).round().astype(np.int16), bite_row, curve["middle"], -1000)
    lab_f, _, _ = cv.to_frame(labels, bite_row, curve["middle"], cv.OUTSIDE)
    info = dict(shape_zyx=list(vol.shape), bite=int(bite), row_shift=int(dr), col_shift=int(dc),
                middle=int(curve["middle"]), n_columns=int(len(curve["px"])))
    return dict(xray=xray_f.astype(np.float16), hu=hu_f, labels=lab_f,
                px=curve["px"], py=curve["py"], nx=curve["nx"], ny=curve["ny"], info=json.dumps(info)), info


def save_preview(ex, path, title):
    """One picture per scan: the X-ray, then two cuts through the block with the labels in color."""
    hu, lab = ex["hu"].astype(np.float32), ex["labels"]
    at_curve = int(round(cv.DEPTH_IN_MM / cv.DEPTH_STEP_MM))  # the depth sample that is on the jaw curve
    row = cv.FRAME_BITE_ROW + int(12 / cv.VOXEL_MM)            # 12 mm below where the teeth meet
    fig, ax = plt.subplots(5, 1, figsize=(10, 14), gridspec_kw=dict(height_ratios=[3, 3, 3, 1, 1]))
    ax[0].imshow(ex["xray"].astype(np.float32), cmap="gray", vmin=0, vmax=1)
    ax[0].set_title(f"{title}: fake X-ray (model input)")
    ax[1].imshow(hu[:, :, at_curve], cmap="gray", vmin=-1000, vmax=2500)
    ax[1].set_title("block, cut along the jaw curve")
    ax[2].imshow(LABEL_COLORS[np.minimum(lab[:, :, at_curve], 5)] * (lab[:, :, at_curve, None] != cv.OUTSIDE))
    ax[2].set_title("labels on the same cut: blue lower jaw, teal upper jaw, red nerve canal, yellow sinus, white teeth")
    ax[3].imshow(hu[row].T, cmap="gray", vmin=-1000, vmax=2500, aspect=cv.DEPTH_STEP_MM / cv.VOXEL_MM)
    ax[3].set_title("block seen from above, 12 mm below the bite (top edge = tongue side, bottom edge = cheek side)")
    ax[4].imshow((LABEL_COLORS[np.minimum(lab[row], 5)] * (lab[row][..., None] != cv.OUTSIDE)).transpose(1, 0, 2),
                 aspect=cv.DEPTH_STEP_MM / cv.VOXEL_MM)
    for a in ax:
        a.axis("off")
    ax[1].axhline(cv.FRAME_BITE_ROW, color="lime", lw=0.5)
    ax[1].axvline(cv.FRAME_MIDDLE_COL, color="lime", lw=0.5)
    plt.tight_layout()
    plt.savefig(path, dpi=60)
    plt.close(fig)


def make_sheets(out, numbers, per_sheet=8):
    """Pictures that show many scans at once: left the X-ray, right the block cut along the jaw curve
    with the labels drawn over it in color. The green lines mark the bite row and the front of the jaw."""
    at_curve = int(round(cv.DEPTH_IN_MM / cv.DEPTH_STEP_MM))
    for start in range(0, len(numbers), per_sheet):
        tiles = []
        for num in numbers[start:start + per_sheet]:
            with np.load(out / f"{num}.npz") as f:
                xray, hu, lab = f["xray"].astype(np.float32), f["hu"][:, :, at_curve].astype(np.float32), f["labels"][:, :, at_curve]
            gray = np.clip((hu + 1000) / 3500, 0, 1)[..., None] * np.ones(3)
            color = LABEL_COLORS[np.minimum(lab, 5)] / 255.0
            marked = (lab > 0) & (lab != cv.OUTSIDE)
            cut = np.where(marked[..., None], 0.55 * gray + 0.45 * color, gray)
            left = xray[..., None] * np.ones(3)
            for img in (left, cut):
                img[cv.FRAME_BITE_ROW, ::4] = (0, 1, 0)
                img[::4, cv.FRAME_MIDDLE_COL] = (0, 1, 0)
            tiles.append(np.concatenate([left, cut], axis=1))
        fig, ax = plt.subplots(len(tiles), 1, figsize=(13.6, 3.2 * len(tiles)), squeeze=False)
        for a, tile, num in zip(ax[:, 0], tiles, numbers[start:start + per_sheet]):
            a.imshow(tile); a.axis("off"); a.set_title(f"scan {num}", fontsize=9, loc="left", pad=2)
        plt.subplots_adjust(left=0, right=1, top=0.985, bottom=0, hspace=0.08)
        plt.savefig(out / "previews" / f"sheet_{start // per_sheet + 1:02d}.png", dpi=80)
        plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="folder that holds imagesTr and labelsTr")
    ap.add_argument("--out", required=True, help="folder for the prepared examples")
    ap.add_argument("--only", nargs="*", help="scan numbers to do, for a quick test (default: all)")
    ap.add_argument("--redo", action="store_true", help="make files again even if they exist")
    args = ap.parse_args()
    out = Path(args.out)
    (out / "previews").mkdir(parents=True, exist_ok=True)
    scans = [s for s in list_scans(args.data) if not args.only or s[0] in args.only]
    print(f"{len(scans)} scans found", flush=True)
    skipped = {}
    for i, (num, scan_path, label_path) in enumerate(scans):
        target = out / f"{num}.npz"
        if target.exists() and not args.redo:
            continue
        t0 = time.time()
        try:
            ex, info = prepare_one(scan_path, label_path)
        except Exception as e:  # one bad scan must not stop the others
            skipped[num] = repr(e)
            target.unlink(missing_ok=True)  # remove a file left by an older run
            print(f"[{i + 1}/{len(scans)}] {num} SKIPPED: {e!r}", flush=True)
            continue
        np.savez_compressed(target, **ex)
        save_preview(ex, out / "previews" / f"{num}.png", f"scan {num}")
        present = [CLASS_NAMES[c] for c in range(1, 6) if (ex["labels"] == c).any()]
        print(f"[{i + 1}/{len(scans)}] {num} done in {time.time() - t0:.0f}s, columns {info['n_columns']}, "
              f"has: {', '.join(present)}", flush=True)
    done = sorted(p.stem for p in out.glob("*.npz"))
    make_sheets(out, done)
    (out / "prepare_report.json").write_text(json.dumps(dict(done=done, skipped=skipped), indent=1))
    print(f"{len(done)} examples are in {out}. Skipped: {skipped if skipped else 'none'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/project/scripts/train.py
"""Train the model that turns a panoramic X-ray into a straightened 3D block.

Run:  python scripts/train.py --data data/prepared --out runs/first --epochs 150

Words used here:
  epoch  = one pass in which the model is shown as many training pieces as there are training scans, times 4.
  step   = the model looks at a few pieces, its output is compared with the correct answer, and its
           numbers are changed a little to make the output closer to the answer.
  loss   = one number that says how far the output is from the correct answer. Lower is better.
  Dice   = overlap between a predicted part and the true part. 0 = no overlap, 1 = the same.

Pause and continue:
  After every epoch the script saves everything it needs in the file last.pt in the --out folder.
  To pause, stop the script (in Colab: the stop button of the cell). To continue, run the same
  command again. It starts after the last saved epoch. At most one epoch of work (about 80 seconds
  on Colab) is lost. The same happens if Colab stops by itself.
"""
import argparse
import csv
import json
import math
import os
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.curved_volume import DEPTH_IN_MM, DEPTH_STEP_MM, FRAME_BITE_ROW, VOXEL_MM
from src.model import XrayTo3D
from src.train_data import OUTSIDE, Examples, change_xray_look, hu_to_unit, make_split, unit_to_hu

CLASS_NAMES = ["background", "lower jaw", "upper jaw", "nerve canal", "sinus", "teeth"]
LABEL_COLORS = np.array([[0, 0, 0], [70, 110, 200], [60, 170, 170], [255, 40, 40], [255, 220, 0], [240, 240, 240]], np.uint8)
# The nerve canal is very small, so mistakes on it count more.
CLASS_WEIGHTS = [1.0, 1.0, 1.0, 4.0, 2.0, 1.0]


def loss_fn(bright, scores, hu, labels):
    """bright [b, d, r, c] 0..1; scores [b, classes, d, r, c]; hu and labels [b, d, r, c]."""
    inside = labels != OUTSIDE
    n_inside = inside.sum().clamp(min=1)
    l_bright = ((bright - hu_to_unit(hu.float())).abs() * inside).sum() / n_inside
    target = labels.long()
    weights = torch.tensor(CLASS_WEIGHTS, device=scores.device, dtype=scores.dtype)
    l_part = F.cross_entropy(scores, target, weight=weights, ignore_index=OUTSIDE)
    # Dice part of the loss: overlap of each predicted part with the true part
    prob = scores.softmax(dim=1)
    misses = []
    for c in range(1, scores.shape[1]):
        true_c = (target == c) & inside
        if true_c.any():
            p = prob[:, c] * inside
            misses.append(1 - (2 * (p * true_c).sum() + 1) / (p.sum() + true_c.sum() + 1))
    l_overlap = torch.stack(misses).mean() if misses else scores.sum() * 0
    return 10 * l_bright + l_part + l_overlap, dict(bright=l_bright.item(), part=l_part.item(), overlap=l_overlap.item())


@torch.no_grad()
def check(model, examples, device, use_half):
    """Run the model on whole scans it was not trained on. Returns average numbers and the last output."""
    model.eval()
    err_hu, psnr, dice = [], [], {c: [] for c in range(1, 6)}
    for i in range(len(examples)):
        xray, hu, labels = (t.to(device) for t in examples.whole(i))
        with torch.autocast(device.type, enabled=use_half):
            bright, scores = model(xray)
        bright, pred = bright.float(), scores.argmax(dim=1)
        inside = labels != OUTSIDE
        diff = (bright - hu_to_unit(hu.float()))[inside]
        err_hu.append(diff.abs().mean().item() * 4000)
        psnr.append(-10 * math.log10(diff.pow(2).mean().item()))
        for c in range(1, 6):
            t, p = (labels == c) & inside, (pred == c) & inside
            if t.any():
                dice[c].append((2 * (t & p).sum() / (t.sum() + p.sum())).item())
    model.train()
    out = dict(err_hu=float(np.mean(err_hu)), psnr=float(np.mean(psnr)))
    for c in range(1, 6):
        out[f"dice {CLASS_NAMES[c]}"] = float(np.mean(dice[c])) if dice[c] else float("nan")
    return out, (xray[0, 0].cpu().numpy(), unit_to_hu(bright[0]).cpu().numpy(), pred[0].cpu().numpy(),
                 hu[0].cpu().numpy(), labels[0].cpu().numpy())


def save_picture(path, title, xray, pred_hu, pred_lab, true_hu, true_lab):
    """Predicted block next to the true block: one cut along the jaw curve and one cut seen from above."""
    outside = true_lab == OUTSIDE  # places outside the scan have no correct answer, so blank them in the output too
    pred_hu, pred_lab = np.where(outside, -1000, pred_hu), np.where(outside, OUTSIDE, pred_lab)
    at_curve = int(round(DEPTH_IN_MM / DEPTH_STEP_MM))  # the depth sample that is on the jaw curve
    row = FRAME_BITE_ROW + int(12 / VOXEL_MM)            # 12 mm below where the teeth meet
    color = lambda lab: LABEL_COLORS[np.minimum(lab, 5)] * (lab[..., None] != OUTSIDE)
    fig, ax = plt.subplots(5, 2, figsize=(18, 17), gridspec_kw=dict(height_ratios=[3, 3, 3, 1.2, 1.2]))
    ax[0, 0].imshow(xray, cmap="gray", vmin=0, vmax=1); ax[0, 0].set_title("input: X-ray")
    ax[0, 1].axis("off"); ax[0, 1].text(0.02, 0.5, title, fontsize=13, va="center")
    for col, (hu, lab, name) in enumerate([(pred_hu, pred_lab, "MODEL OUTPUT"), (true_hu, true_lab, "CORRECT ANSWER")]):
        ax[1, col].imshow(hu[at_curve], cmap="gray", vmin=-1000, vmax=2500); ax[1, col].set_title(f"{name}: cut along the jaw curve")
        ax[2, col].imshow(color(lab[at_curve])); ax[2, col].set_title(f"{name}: parts (blue lower jaw, teal upper jaw, red nerve canal, yellow sinus, white teeth)")
        ax[3, col].imshow(hu[:, row], cmap="gray", vmin=-1000, vmax=2500, aspect=1.25); ax[3, col].set_title(f"{name}: seen from above, 12 mm below the bite")
        ax[4, col].imshow(color(lab[:, row]), aspect=1.25)
    for a in ax.ravel():
        a.axis("off")
    plt.tight_layout(); plt.savefig(path, dpi=55); plt.close(fig)


PROGRESS = dict(saved_epochs=0, total=0)  # used for the message that is printed when the script is paused


def save_safely(state, path):
    """Write to a second file first, then rename it. If the script is stopped in the middle of
    writing, the old file is still whole."""
    tmp = Path(str(path) + ".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="folder made by prepare_tf3.py")
    ap.add_argument("--out", required=True, help="folder for saved models, the log and pictures")
    ap.add_argument("--epochs", type=int, default=150)
    ap.add_argument("--batch", type=int, default=2, help="pieces per step")
    ap.add_argument("--piece-cols", type=int, default=384, help="width of one training piece")
    ap.add_argument("--lr", type=float, default=5e-4, help="how big each change to the model is")
    ap.add_argument("--limit", type=int, default=0, help="use only this many training scans (for a quick test)")
    ap.add_argument("--small", action="store_true", help="use a much smaller model (only for a quick test on a laptop)")
    args = ap.parse_args()

    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_half = device.type == "cuda"  # half-size numbers on the graphics card: faster and uses less memory
    print(f"running on: {device}" + (f" ({torch.cuda.get_device_name(0)})" if use_half else ""), flush=True)

    split = make_split(args.data)
    train_nums = split["train"][:args.limit] if args.limit else split["train"]
    check_nums = split["check"][:max(1, args.limit)] if args.limit else split["check"]
    t0 = time.time()
    train, checks = Examples(args.data, train_nums), Examples(args.data, check_nums)
    print(f"loaded {len(train)} training scans and {len(checks)} check scans in {time.time() - t0:.0f}s", flush=True)

    depth = train.hu.shape[1]
    model = XrayTo3D(depth=depth, widths=(16, 32, 48, 64, 64) if args.small else (48, 96, 192, 384, 512)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(enabled=use_half)
    steps_per_epoch = max(1, len(train) * 4 // args.batch)
    start, best = 0, -1.0
    last_path = out / "last.pt"
    if last_path.exists():
        saved = torch.load(last_path, map_location=device, weights_only=False)  # our own file, so it is safe to load fully
        model.load_state_dict(saved["model"]); opt.load_state_dict(saved["opt"])
        if "scaler" in saved:
            scaler.load_state_dict(saved["scaler"])
        start, best = saved["epoch"], saved["best"]
        print(f"found saved work: {start} of {args.epochs} epochs are finished. Continuing with epoch {start + 1}.", flush=True)
    else:
        print("no saved work found in this folder. Starting at epoch 1.", flush=True)
    PROGRESS.update(saved_epochs=start, total=args.epochs)
    (out / "settings.json").write_text(json.dumps(dict(vars(args), train=train_nums, check=check_nums), indent=1))

    for epoch in range(start, args.epochs):
        lr = args.lr * 0.5 * (1 + math.cos(math.pi * epoch / args.epochs))  # start big, end small
        for g in opt.param_groups:
            g["lr"] = max(lr, 1e-5)
        t0, sums = time.time(), {}
        for _ in range(steps_per_epoch):
            xray, hu, labels = (t.to(device) for t in train.random_batch(args.batch, args.piece_cols))
            xray = change_xray_look(xray)
            with torch.autocast(device.type, enabled=use_half):
                bright, scores = model(xray)
            loss, parts = loss_fn(bright.float(), scores.float(), hu, labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            parts["loss"] = loss.item()
            for k, v in parts.items():
                sums[k] = sums.get(k, 0.0) + v
        row = dict(epoch=epoch + 1, seconds=round(time.time() - t0), **{f"train {k}": v / steps_per_epoch for k, v in sums.items()})
        res, shown = check(model, checks, device, use_half)
        row.update(res)
        score = float(np.nanmean([res[f"dice {CLASS_NAMES[c]}"] for c in range(1, 6)]))
        if use_half:
            row["gpu_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 1)
        print(f"epoch {epoch + 1}/{args.epochs} ({row['seconds']}s" + (f", {row['gpu_gb']} GB of graphics card memory" if use_half else "") + f") loss {row['train loss']:.3f} | check scans: "
              f"error {res['err_hu']:.0f} HU, PSNR {res['psnr']:.1f}, Dice " +
              ", ".join(f"{CLASS_NAMES[c]} {res[f'dice {CLASS_NAMES[c]}']:.2f}" for c in range(1, 6)), flush=True)
        new_file = not (out / "log.csv").exists()
        with open(out / "log.csv", "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            if new_file:
                w.writeheader()
            w.writerow(row)
        state = dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), epoch=epoch + 1,
                     best=max(best, score), depth=depth, small=args.small)
        if score > best:
            best = score
            save_safely(state, out / "best.pt")
        save_safely(state, last_path)
        PROGRESS["saved_epochs"] = epoch + 1
        if (epoch + 1) % 10 == 0 or epoch + 1 == args.epochs or epoch == 0:
            save_picture(out / f"check_epoch_{epoch + 1:03d}.png", f"scan {checks.numbers[-1]} (not used for training)\nafter epoch {epoch + 1}", *shown)
    print(f"finished. Best average Dice on check scans: {best:.3f}. Files are in {out}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:  # the stop button
        print(f"\nPAUSED. {PROGRESS['saved_epochs']} of {PROGRESS['total']} epochs are finished and saved. "
              "To continue, run this same cell (or command) again.", flush=True)


The next cell trains the model. After every epoch it saves the model in the results folder.

In [ ]:
!python /kaggle/working/project/scripts/train.py --data /tmp/prepared --out "$OUT" --epochs $EPOCHS --limit $TRAIN_SCANS

In [ ]:
import glob, pandas as pd
from IPython.display import Image, display
display(pd.read_csv(OUT + "/log.csv").round(3).tail(10))
display(Image(sorted(glob.glob(OUT + "/check_epoch_*.png"))[-1], width=1100))